# 06_B_tuning — Optuna, LightGBM, calibration (val only; test is not touched)
1. CatBoost: Optuna, **30 trials**, `GroupKFold` by EXPID inside train, objective = **log loss**, `iterations <= 500`, early stopping 50, `thread_count=-1`.
2. LightGBM with native categorical features: **12 trials**, same CV and objective, for comparison.
3. Best model (by val log loss) is calibrated with **Platt** and **isotonic**. Val expeditions are split in two random halves; each half is calibrated with the calibrator learned on the other half, so every val row gets an out-of-fold calibrated probability. Calibrators are learned on non-anomalous rows only.
4. Calibration curves and Brier before/after; everything logged.

Expect CatBoost tuning to take a while (30 trials x 5 folds); set `N_TRIALS_CB = 5` for a dry run.

In [1]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

from src.split import load_features, dev_split, fit_frame, TARGET
from src.evaluate import summary_row, results_table, plot_calibration, metrics_by_group
from src import models as M
from src.experiment_log import log_experiment

N_BOOT = 1000   # bootstrap resamples of whole expeditions (lower to ~200 for a quick dry run)
FIG = ROOT / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)

In [2]:
df = load_features()
train, val = dev_split(df)              # the TEST rows are never loaded in this notebook
train_fit = fit_frame(train)            # fitting rows: anomalous seasons dropped
clean = ~val["ANOMALOUS_SEASON"].to_numpy()   # val view without anomalous seasons (2020, spring 2021)
print(f"train {len(train):,} (fit {len(train_fit):,}) | val {len(val):,} (clean {clean.sum():,})")
print(f"{TARGET}: train {train[TARGET].mean():.3f} | val {val[TARGET].mean():.3f} | val clean {val.loc[clean, TARGET].mean():.3f}")
print(f"model features ({len(M.MODEL_FEATURES)}): YEAR excluded; categorical: {M.cat_names(M.MODEL_FEATURES)}")

train 37,969 (fit 37,448) | val 5,042 (clean 4,236)
MSUCCESS: train 0.383 | val 0.537 | val clean 0.549
model features (34): YEAR excluded; categorical: ['SEX', 'CITIZEN', 'STATUS', 'PEAK_GROUP', 'ROUTE1', 'SEASON', 'HOST', 'AGENCY', 'HIMAL', 'REGION']


In [3]:
def evaluate_all(preds):
    """Two views: all val rows, and val without anomalous seasons."""
    all_rows = [summary_row(n, val[TARGET], p, val["EXPID"], N_BOOT) for n, p in preds.items()]
    clean_rows = [summary_row(n, val.loc[clean, TARGET], p[clean], val.loc[clean, "EXPID"], N_BOOT) for n, p in preds.items()]
    return all_rows, clean_rows

In [4]:
N_TRIALS_CB, N_TRIALS_LGBM = 30, 12 #the initial tesing run was on 5,5

## 1. CatBoost + Optuna

In [ ]:
study_cb = M.tune_catboost(train_fit, n_trials=N_TRIALS_CB, metric="logloss")
trials_cb = study_cb.trials_dataframe(attrs=("number", "value", "params", "user_attrs"))
trials_cb.to_csv(ROOT / "reports" / "06_B_optuna_catboost_trials.csv", index=False)
print("best CV log loss:", round(study_cb.best_value, 4)); print(study_cb.best_params)
display(trials_cb.sort_values("value").head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(trials_cb["number"], trials_cb["value"], "o-", ms=3); ax.plot(trials_cb["number"], trials_cb["value"].cummin(), "r-")
ax.set_xlabel("trial"); ax.set_ylabel("CV log loss"); ax.set_title("CatBoost Optuna history")
plt.tight_layout(); plt.savefig(FIG / "06_B_optuna_catboost_history.png", dpi=150); plt.show()

In [ ]:
cb_tuned = M.fit_catboost(train_fit, params=M.catboost_params_from(study_cb))
p_cb = cb_tuned.predict(val)
cb_default = M.fit_catboost(train_fit); p_cb_default = cb_default.predict(val)

## 2. LightGBM (categorical features) for comparison

In [ ]:
study_lgbm = M.tune_lightgbm(train_fit, n_trials=N_TRIALS_LGBM, metric="logloss")
trials_lgbm = study_lgbm.trials_dataframe(attrs=("number", "value", "params", "user_attrs"))
trials_lgbm.to_csv(ROOT / "reports" / "06_B_optuna_lightgbm_trials.csv", index=False)
print("best CV log loss:", round(study_lgbm.best_value, 4)); print(study_lgbm.best_params)
lgbm = M.fit_lightgbm(train_fit, params={**study_lgbm.best_params, "subsample_freq": 1})
p_lgbm = lgbm.predict(val)

## 3. Compare on val

In [ ]:
preds = {"CatBoost default": p_cb_default, "CatBoost tuned (Optuna 30)": p_cb, "LightGBM tuned (Optuna 12)": p_lgbm}
rows_all, rows_clean = evaluate_all(preds)
rows_models = (rows_all, rows_clean)
print("VAL, all rows"); display(results_table(rows_all))
print("VAL, without anomalous seasons"); display(results_table(rows_clean))
best = min(("CatBoost tuned (Optuna 30)", "LightGBM tuned (Optuna 12)"), key=lambda n: next(r for r in rows_all if r["model"] == n)["_res"]["log_loss"]["estimate"])
print("best tuned model by val log loss:", best)

## 4. Calibration of the best model (Platt and isotonic, out-of-fold on val halves)

In [ ]:
p_best = preds[best]
fit_rows = clean                                    # learn calibrators on non-anomalous rows only
cal = {"before": p_best,
       "Platt": M.crossfit_calibrate(val, p_best, "platt", fit_rows=fit_rows),
       "isotonic": M.crossfit_calibrate(val, p_best, "isotonic", fit_rows=fit_rows)}
rows_all_c, rows_clean_c = evaluate_all({f"{best} | {k}": v for k, v in cal.items()})
print("VAL, all rows"); display(results_table(rows_all_c))
print("VAL, without anomalous seasons"); display(results_table(rows_clean_c))
for view, rows in (("all", rows_all_c), ("clean", rows_clean_c)):
    b = [r["_res"]["brier"]["estimate"] for r in rows]
    print(f"Brier ({view}): before {b[0]:.4f} | Platt {b[1]:.4f} | isotonic {b[2]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_calibration(val[TARGET], cal, ax=axes[0], title=f"{best}: all val rows")
plot_calibration(val.loc[clean, TARGET], {k: v[clean] for k, v in cal.items()}, ax=axes[1], title="val without anomalous seasons")
plt.tight_layout(); plt.savefig(FIG / "06_B_calibration_before_after.png", dpi=150); plt.show()

## 5. Log everything

In [ ]:
every = {**{r["model"]: (r["_res"], c["_res"]) for r, c in zip(*rows_models)},
         **{r["model"]: (r["_res"], c["_res"]) for r, c in zip(rows_all_c, rows_clean_c)}}
notes = {"CatBoost default": "Default params; early stopping inside train.",
         "CatBoost tuned (Optuna 30)": f"Optuna 30 trials, grouped 5-fold CV on train, log loss; best CV {study_cb.best_value:.4f}.",
         "LightGBM tuned (Optuna 12)": f"Optuna 12 trials, native categoricals; best CV {study_lgbm.best_value:.4f}."}
for name, (ra, rc) in every.items():
    note = notes.get(name, "Calibrated on the other half of val (out-of-fold), anomalous rows excluded from calibrator fit.")
    log_experiment("B " + name, ra, rc, note)
print("README.md and reports/experiment_log.csv updated")